# Pairing Generation

---

## Techniques Used

1. Dynamic Programming
2. Depth-first Search Algorithm
3. Nearest Neighbor Algorithm

---

---

### Importing the Packages

---

In [2]:
# importing the packages
import pandas as pd
from itertools import permutations
import numpy as np

---

### Reading the Data to a DataFrame

---

In [3]:
# reading the dataframe
df = pd.read_csv("../data/data.csv")
df = df.head(30)

---

## 1. Dynamic Programming

---

In [4]:
# create all possible pairs of flight_leg using combinations
flight_leg_combinations = list(permutations(df['flight_leg_id'], 3))



# Filter pairs based on the specified condition
filtered_combinations = [pair for pair in flight_leg_combinations if 
                         df.loc[df['flight_leg_id'] == pair[0], 'destination_airport'].values[0] ==
                         df.loc[df['flight_leg_id'] == pair[2], 'departure_airport'].values[0]]

# Filter pairs based on the specified condition
filtered_combinations_1 = [pair for pair in filtered_combinations if 
                         (df.loc[df['flight_leg_id'] == pair[1], 'start_time'].values[0] - 
                          df.loc[df['flight_leg_id'] == pair[0], 'end_time'].values[0]) >= 1]

filtered_combinations_2 = [pair for pair in filtered_combinations_1 if 
                         (df.loc[df['flight_leg_id'] == pair[2], 'start_time'].values[0] - 
                          df.loc[df['flight_leg_id'] == pair[1], 'end_time'].values[0]) >= 1]

# create an empty dataframe for the pair matrix
pair_matrix = pd.DataFrame(index=range(1, len(filtered_combinations_2) + 1), columns=df['flight_leg_id'])

# fill the pair matrix based on flight leg combinations
for i, pair in enumerate(filtered_combinations_2):
    pair_matrix.loc[i+1, pair[0]] = 1
    pair_matrix.loc[i+1, pair[1]] = 1
    pair_matrix.loc[i+1, pair[2]] = 1

# fill NaN values with 0
pair_matrix = pair_matrix.fillna(0)

---

---

---

## 2. Depth-first Search Algorithm

---

In [12]:
def generate_pairs_dfs(df, current_pair, valid_pairs):
    # Base case: If the current pair is complete, add it to the list of valid pairs
    if len(current_pair) == 3:
        valid_pairs.append(current_pair.copy())
        return

    # Recursive DFS for each remaining flight leg
    for flight_leg_id in df['flight_leg_id']:
        # Check if the leg can be added to the current pair
        if is_valid_pair(df, current_pair, flight_leg_id):
            current_pair.append(flight_leg_id)
            generate_pairs_dfs(df, current_pair, valid_pairs)
            current_pair.pop()  # Backtrack

def is_valid_pair(df, current_pair, flight_leg_id):
    # Implement conditions for pair validity
    # Ensure matching destination and departure airports, and time gaps
    return (
        len(current_pair) == 0 or
        (df.loc[df['flight_leg_id'] == current_pair[-1], 'destination_airport'].values[0] ==
         df.loc[df['flight_leg_id'] == flight_leg_id, 'departure_airport'].values[0]) and
        (df.loc[df['flight_leg_id'] == flight_leg_id, 'start_time'].values[0] -
         df.loc[df['flight_leg_id'] == current_pair[-1], 'end_time'].values[0]) >= 1
    )

# Initialize
valid_pairs_dfs = []

# Start DFS from each flight leg
for start_leg in df['flight_leg_id']:
    generate_pairs_dfs(df, [start_leg], valid_pairs_dfs)

# Create an empty dataframe for the pair matrix
pair_matrix_dfs = pd.DataFrame(index=range(1, len(valid_pairs_dfs) + 1), columns=df['flight_leg_id'])

# Fill the pair matrix based on flight leg combinations
for i, pair in enumerate(valid_pairs_dfs):
    for leg in pair:
        pair_matrix_dfs.loc[i+1, leg] = 1

# Fill NaN values with 0
pair_matrix_dfs = pair_matrix_dfs.fillna(0)

# Display the result
pair_matrix_dfs


flight_leg_id,1,2,3,4,5,6,7,8,9,10,...,91,92,93,94,95,96,97,98,99,100
1,0,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5,0,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2475,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2476,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2477,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2478,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


---  

---

---

## Nearest Neighbor

---

In [6]:
def generate_pairs_nearest_neighbor(df):
    valid_pairs = []
    visited_legs = set()

    for start_leg in df['flight_leg_id']:
        current_pair = [start_leg]
        visited_legs.add(start_leg)

        while len(current_pair) < 3:
            nearest_neighbor = find_nearest_neighbor(df, current_pair, visited_legs)
            
            if nearest_neighbor is None:
                break
            
            current_pair.append(nearest_neighbor)
            visited_legs.add(nearest_neighbor)

        if len(current_pair) == 3:
            valid_pairs.append(current_pair.copy())

    return valid_pairs

def find_nearest_neighbor(df, current_pair, visited_legs):
    # Find the nearest neighbor based on specified criteria
    current_leg = current_pair[-1]
    available_legs = set(df['flight_leg_id']) - visited_legs
    min_distance = np.inf
    nearest_neighbor = None

    for leg in available_legs:
        if is_valid_pair(df, current_leg, leg):
            distance = calculate_distance(df, current_leg, leg)
            if distance < min_distance:
                min_distance = distance
                nearest_neighbor = leg

    return nearest_neighbor

def is_valid_pair(df, leg1, leg2):
    # Implement conditions for pair validity
    # Ensure matching destination and departure airports, and time gaps
    return (
        (df.loc[df['flight_leg_id'] == leg1, 'destination_airport'].values[0] ==
         df.loc[df['flight_leg_id'] == leg2, 'departure_airport'].values[0]) and
        (df.loc[df['flight_leg_id'] == leg2, 'start_time'].values[0] -
         df.loc[df['flight_leg_id'] == leg1, 'end_time'].values[0]) >= 1
    )

def calculate_distance(df, leg1, leg2):
    # Calculate a distance metric (e.g., time difference) between two flight legs
    return abs(df.loc[df['flight_leg_id'] == leg1, 'end_time'].values[0] -
               df.loc[df['flight_leg_id'] == leg2, 'start_time'].values[0])


# Run the Nearest Neighbor algorithm
valid_pairs_nn = generate_pairs_nearest_neighbor(df)

# Create an empty dataframe for the pair matrix
pair_matrix_nn = pd.DataFrame(index=range(1, len(valid_pairs_nn) + 1), columns=df['flight_leg_id'])

# Fill the pair matrix based on flight leg combinations
for i, pair in enumerate(valid_pairs_nn):
    for leg in pair:
        pair_matrix_nn.loc[i+1, leg] = 1

# Fill NaN values with 0
pair_matrix_nn = pair_matrix_nn.fillna(0)

# Display the result
pair_matrix_nn


flight_leg_id,1,2,3,4,5,6,7,8,9,10,...,41,42,43,44,45,46,47,48,49,50
1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1


---

---

---

### Cost of Pairings

---

In [9]:
# Creating the cost matrix from the final filtered combinations
cost_list = [
    (
        df.loc[df['flight_leg_id'] == pair[2], 'end_time'].values[0] -
        df.loc[df['flight_leg_id'] == pair[0], 'start_time'].values[0]
    )
    for pair in filtered_combinations_2
]


In [10]:
cost_list

[20,
 19,
 20,
 15,
 20,
 19,
 20,
 20,
 19,
 20,
 13,
 15,
 20,
 19,
 20,
 13,
 15,
 20,
 19,
 11,
 20,
 20,
 19,
 20,
 20,
 19,
 20,
 15,
 20,
 19,
 20,
 13,
 15,
 20,
 19,
 11,
 20,
 20,
 19,
 20,
 20,
 19,
 20,
 15,
 20,
 19,
 20,
 13,
 15,
 20,
 19,
 11,
 20,
 9,
 15,
 20,
 19,
 20,
 13,
 15,
 8,
 20,
 19,
 11,
 20,
 9,
 20,
 19,
 20,
 13,
 15,
 20,
 19,
 11,
 20,
 15,
 20,
 19,
 20,
 15,
 20,
 19,
 20,
 20,
 13,
 15,
 20,
 19,
 11,
 20,
 13,
 15,
 20,
 19,
 20,
 13,
 15,
 20,
 19,
 20,
 13,
 15,
 20,
 19,
 11,
 20,
 13,
 15,
 20,
 19,
 20,
 8,
 9,
 8,
 8,
 8,
 9,
 8,
 8,
 9,
 8,
 8,
 8,
 9,
 8,
 9,
 18,
 18,
 18,
 18,
 9,
 18,
 18,
 18,
 18,
 9,
 18,
 18,
 18,
 18,
 9,
 18,
 18,
 9,
 18,
 18,
 9,
 18,
 18,
 18,
 9,
 18,
 18,
 18,
 9,
 18,
 18,
 11,
 8,
 9,
 11,
 8,
 11,
 12,
 9,
 10,
 12,
 6,
 8,
 9,
 10,
 12,
 6,
 8,
 9,
 10,
 12,
 9,
 12,
 9,
 10,
 12,
 9,
 10,
 12,
 9,
 12,
 15,
 14,
 15,
 10,
 15,
 14,
 15,
 15,
 14,
 15,
 8,
 10,
 15,
 14,
 15,
 15,
 14,
 15,
 15,
 14,
 15,


---

---